In [1]:
import pandas as pd
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("image_data_prep").getOrCreate()

In [2]:
from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, FloatType, IntegerType
from PIL import Image
import numpy as np
import io

# Initialize Spark Session
spark = SparkSession.builder.appName("image_data_prep").getOrCreate()

# Define function to read and preprocess images
def preprocess_image(binary_data):
    try:
        if binary_data is None:
            return None
        img = Image.open(io.BytesIO(bytearray(binary_data))).convert('L')  # Convert to grayscale
        img = img.resize((128, 128))  # Resize for uniformity
        img_array = np.array(img).flatten().tolist()  # Flatten to 1D array
        return img_array
    except Exception as e:
        print(f"Error processing image: {str(e)}")
        return None

# Define UDF for preprocessing
preprocess_udf = udf(preprocess_image, ArrayType(FloatType()))

# Define image directories
test_data_no_loc = "hdfs://localhost:9000/user/jj/final_proj/Brain_Tumor_Datasets/test/no"
test_data_yes_loc = "hdfs://localhost:9000/user/jj/final_proj/Brain_Tumor_Datasets/test/yes"
train_data_no_loc = "hdfs://localhost:9000/user/jj/final_proj/Brain_Tumor_Datasets/train/no"
train_data_yes_loc = "hdfs://localhost:9000/user/jj/final_proj/Brain_Tumor_Datasets/train/yes"

# Load data using Spark Image Data Source
test_data_no = spark.read.format("image").load(test_data_no_loc)
test_data_yes = spark.read.format("image").load(test_data_yes_loc)
train_data_no = spark.read.format("image").load(train_data_no_loc)
train_data_yes = spark.read.format("image").load(train_data_yes_loc)

# Apply preprocessing
test_data_no = test_data_no.withColumn("features", preprocess_udf(test_data_no["image.data"])).withColumn("label", test_data_no["image.height"] * 0)
test_data_yes = test_data_yes.withColumn("features", preprocess_udf(test_data_yes["image.data"])).withColumn("label", test_data_yes["image.height"] * 0 + 1)
train_data_no = train_data_no.withColumn("features", preprocess_udf(train_data_no["image.data"])).withColumn("label", train_data_no["image.height"] * 0)
train_data_yes = train_data_yes.withColumn("features", preprocess_udf(train_data_yes["image.data"])).withColumn("label", train_data_yes["image.height"] * 0 + 1)

# Combine datasets
df = test_data_no.union(test_data_yes).union(train_data_no).union(train_data_yes)

df.count()

+--------------------+--------+-----+
|               image|features|label|
+--------------------+--------+-----+
|{hdfs://localhost...|    NULL|    0|
|{hdfs://localhost...|    NULL|    0|
|{hdfs://localhost...|    NULL|    0|
|{hdfs://localhost...|    NULL|    0|
|{hdfs://localhost...|    NULL|    0|
+--------------------+--------+-----+
only showing top 5 rows



In [3]:
import seaborn as sns

In [ ]:
df_pandas = df.toPandas()


In [ ]:
sns.countplot(data = df_pandas, x = 'label')